# 03 — Feature engineering (z-score-safe) + greedy selection
Band powers are z-scored, so ~50% of values are <= 0. Raw ratios would divide by near-zero and flip sign. We use **differences/sums** and **softplus ratios** instead. We add one feature group at a time and keep it only if it improves OOF macro-F1 by a margin larger than the per-fold noise (>0.002).

In [1]:
# --- Shared toolbox (identical across notebooks; see build_notebooks.py) ---
import warnings, json
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, classification_report, confusion_matrix

SEED = 42
N_FOLDS = 5
CLASS_NAMES = {0: "Wake", 1: "Light", 2: "Deep", 3: "REM"}
CLASSES = np.array([0, 1, 2, 3])
EOG = "eog_burst_index"            # the only column with missing values (~50%)

RAW_FEATURES = [
    "eeg_delta_power", "eeg_theta_power", "eeg_alpha_power", "eeg_sigma_power",
    "eeg_beta_power", "eeg_gamma_power", "eeg_slow_osc_power", "eeg_spectral_entropy",
    "eeg_spindle_density", "eeg_kcomplex_rate", "emg_chin_tone", "emg_tone_variance",
    "eog_movement_density", "eog_amplitude", "heart_rate_mean", "heart_rate_variability",
    "respiration_rate", "respiration_variability", "spo2_mean", "body_movement_index",
    EOG,
]
POWER = ["eeg_delta_power", "eeg_theta_power", "eeg_alpha_power", "eeg_sigma_power",
         "eeg_beta_power", "eeg_gamma_power", "eeg_slow_osc_power"]

HERE = Path.cwd()
ART = HERE / "artifacts"; ART.mkdir(exist_ok=True)
SUB = HERE / "submissions"; SUB.mkdir(exist_ok=True)


def load_data():
    """Return (train_df, test_df). Features kept as-is (NaN preserved)."""
    tr = pd.read_csv(HERE / "train.csv")
    te = pd.read_csv(HERE / "test.csv")
    return tr, te


def macro_f1(y_true, y_pred):
    """Competition metric: macro-averaged F1 over the 4 classes."""
    return f1_score(y_true, y_pred, average="macro")


def per_class_f1(y_true, y_pred):
    f = f1_score(y_true, y_pred, average=None, labels=CLASSES)
    return {CLASS_NAMES[c]: round(float(f[i]), 4) for i, c in enumerate(CLASSES)}


def softplus(x):
    """Numerically stable log(1+exp(x)); strictly positive and monotonic.
    Used to turn z-scored band powers (~50% negative) into positive magnitudes
    so band ratios are well-defined instead of dividing by near-zero."""
    x = np.asarray(x, dtype=float)
    return np.log1p(np.exp(-np.abs(x))) + np.maximum(x, 0.0)


def _aligned_proba(model, X):
    """predict_proba with columns aligned to CLASSES = [0,1,2,3]."""
    p = model.predict_proba(X)
    cls = list(np.asarray(model.classes_))
    idx = [cls.index(c) for c in CLASSES]
    return p[:, idx]


def run_oof(make_model, X, y, X_test, folds, needs_impute=False, use_eval_set=False):
    """Out-of-fold training under fixed folds.

    Returns (oof, test_p, oof_macro, fold_scores):
      oof     : (n_train, 4) out-of-fold probabilities (each row predicted once)
      test_p  : (n_test, 4) test probabilities, averaged over the 5 fold-models
      oof_macro: global macro-F1 over the assembled OOF matrix (primary metric)

    Two model families, identical folds:
      - CatBoost (needs_impute=False): NaN passed through natively.
      - sklearn trees (needs_impute=True): add EOG-missing flag + fill EOG NaN
        with the TRAIN-FOLD median (fit on train fold only -> no leakage)."""
    n = len(y)
    oof = np.zeros((n, len(CLASSES)))
    test_p = np.zeros((len(X_test), len(CLASSES)))
    fold_scores = []
    for tr_idx, va_idx in folds:
        Xtr, Xva, Xte = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy(), X_test.copy()
        ytr, yva = y[tr_idx], y[va_idx]
        if needs_impute:
            med = Xtr[EOG].median()
            for d in (Xtr, Xva, Xte):
                if EOG + "_missing" not in d.columns:
                    d[EOG + "_missing"] = d[EOG].isna().astype("int8")
                d[EOG] = d[EOG].fillna(med)
            assert not Xtr.isna().any().any(), "NaN remained after impute"
        model = make_model()
        if use_eval_set:
            model.fit(Xtr, ytr, eval_set=(Xva, yva))
        else:
            model.fit(Xtr, ytr)
        oof[va_idx] = _aligned_proba(model, Xva)
        test_p += _aligned_proba(model, Xte) / len(folds)
        fold_scores.append(macro_f1(yva, oof[va_idx].argmax(1)))
    oof_macro = macro_f1(y, oof.argmax(1))
    return oof, test_p, oof_macro, fold_scores


def load_folds():
    """Load the fixed StratifiedKFold split saved by 02_baseline."""
    d = np.load(ART / "folds.npz", allow_pickle=True)
    return [(d[f"tr{i}"], d[f"va{i}"]) for i in range(N_FOLDS)]


In [2]:
# --- z-score-safe feature engineering (single source of truth) ---
EPS = 1e-3

def add_features(df, groups):
    """Return RAW_FEATURES plus the requested engineered feature groups.

    Features are z-scored, so ~50% of every band-power value is <= 0. Raw ratios
    (delta/theta, ...) would divide by near-zero and flip sign -> meaningless.
    We therefore use (a) differences/sums (well-defined for z-scores) and
    (b) softplus-transformed ratios (positive magnitudes). The EOG channel is
    ~50% missing; we never derive features from it (only a missing-indicator)."""
    X = df[RAW_FEATURES].copy()
    miss = df[EOG].isna().astype("int8")          # informative channel on/off

    if "missing" in groups:
        X[EOG + "_missing"] = miss

    if "contrast" in groups:
        d, th, al, sig, be, ga, so = (df["eeg_delta_power"], df["eeg_theta_power"],
            df["eeg_alpha_power"], df["eeg_sigma_power"], df["eeg_beta_power"],
            df["eeg_gamma_power"], df["eeg_slow_osc_power"])
        X["delta_minus_theta"] = d - th
        X["theta_minus_alpha"] = th - al
        X["slowosc_minus_delta"] = so - d
        X["beta_minus_delta"] = be - d
        X["dt_minus_ab"] = (d + th) - (al + be)
        X["eeg_total"] = df[POWER].sum(axis=1)

    if "ratio" in groups:
        sp = {c: softplus(df[c]) for c in POWER}
        X["r_delta_theta"] = sp["eeg_delta_power"] / (sp["eeg_theta_power"] + EPS)
        X["r_theta_alpha"] = sp["eeg_theta_power"] / (sp["eeg_alpha_power"] + EPS)
        X["r_slowosc_delta"] = sp["eeg_slow_osc_power"] / (sp["eeg_delta_power"] + EPS)
        X["r_beta_delta"] = sp["eeg_beta_power"] / (sp["eeg_delta_power"] + EPS)
        X["r_dt_ab"] = (sp["eeg_delta_power"] + sp["eeg_theta_power"]) / (
            sp["eeg_alpha_power"] + sp["eeg_beta_power"] + EPS)

    if "relpower" in groups:
        sp = {c: softplus(df[c]) for c in POWER}
        tot = sum(sp.values()) + EPS
        for c in POWER:
            X["frac_" + c.replace("eeg_", "").replace("_power", "")] = sp[c] / tot

    if "autonomic" in groups:
        X["hr_over_resp"] = df["heart_rate_mean"] / (softplus(df["respiration_rate"]) + EPS)
        X["hrv_x_respvar"] = df["heart_rate_variability"] * df["respiration_variability"]
        X["move_x_emg"] = df["body_movement_index"] * df["emg_chin_tone"]

    if "eog" in groups:
        X["eog_amp_x_missing"] = df["eog_amplitude"] * miss
        X["eog_move_x_missing"] = df["eog_movement_density"] * miss

    return X


In [3]:
from catboost import CatBoostClassifier
train, test = load_data()
y = np.load(ART / "y_train.npy")
folds = load_folds()

# Use the SAME full-power CatBoost as the final model so feature decisions
# match what we actually deploy (a cheap proxy misjudged the missing flag).
# Early stopping keeps it bounded.
def make_cat():
    return CatBoostClassifier(loss_function="MultiClass",
        eval_metric="TotalF1:average=Macro", iterations=3000, learning_rate=0.03,
        depth=6, l2_leaf_reg=3.0, random_seed=SEED, od_type="Iter", od_wait=150,
        use_best_model=True, allow_writing_files=False, thread_count=-1, verbose=False)

def score_groups(groups):
    Xtr = add_features(train, groups); Xte = add_features(test, groups)
    _, _, m, _ = run_oof(make_cat, Xtr, y, Xte, folds,
                         needs_impute=False, use_eval_set=True)
    return m

## Greedy forward selection
The EOG missing-indicator is a standard, near-free treatment for *informative* missingness (and the baseline already showed it helps), so we always keep it. We greedily test only the heavier engineered groups, keeping one only if it beats the per-fold noise (>0.002).

In [4]:
BASE_GROUPS = ["missing"]                       # always kept (informative missingness)
CAND_GROUPS = ["contrast", "ratio", "relpower", "autonomic", "eog"]
MARGIN = 0.002
kept = list(BASE_GROUPS)
best = score_groups(kept)
print(f"base (raw 21 + missing flag): OOF macro-F1 = {best:.5f}")
trials = [("raw21+missing", best, "base")]
for g in CAND_GROUPS:
    cand = score_groups(kept + [g])
    keep = cand > best + MARGIN
    verdict = "KEEP" if keep else "drop"
    trials.append((g, cand, verdict))
    print(f"  + {g:<10} -> {cand:.5f}  ({cand-best:+.5f})  {verdict}")
    if keep:
        kept.append(g); best = cand
print("\nkept groups:", kept, "| best OOF macro-F1:", round(best, 5))

base (raw 21 + missing flag): OOF macro-F1 = 0.82494


  + contrast   -> 0.82898  (+0.00404)  KEEP


  + ratio      -> 0.82541  (-0.00357)  drop


  + relpower   -> 0.82535  (-0.00363)  drop


  + autonomic  -> 0.82741  (-0.00156)  drop


  + eog        -> 0.82032  (-0.00865)  drop

kept groups: ['missing', 'contrast'] | best OOF macro-F1: 0.82898


## Save the chosen feature set (`features_v1`)
Engineered train+test matrices (NaN preserved in `eog_burst_index`) + the column list, so downstream notebooks reload exactly these features.

In [5]:
Xtr_v1 = add_features(train, kept)
Xte_v1 = add_features(test, kept)
feature_cols = list(Xtr_v1.columns)
np.save(ART / "features_v1_train.npy", Xtr_v1.values.astype("float64"))
np.save(ART / "features_v1_test.npy", Xte_v1.values.astype("float64"))
json.dump({"groups": kept, "columns": feature_cols},
          open(ART / "feature_cols.json", "w"), indent=2)
print(f"features_v1: {len(feature_cols)} columns")
print(feature_cols)
# sanity: no inf, and NaN only in the EOG column
assert not np.isinf(Xtr_v1.select_dtypes("number").values).any()
na_cols = Xtr_v1.columns[Xtr_v1.isna().any()].tolist()
print("columns with NaN (expected just EOG):", na_cols)

features_v1: 28 columns
['eeg_delta_power', 'eeg_theta_power', 'eeg_alpha_power', 'eeg_sigma_power', 'eeg_beta_power', 'eeg_gamma_power', 'eeg_slow_osc_power', 'eeg_spectral_entropy', 'eeg_spindle_density', 'eeg_kcomplex_rate', 'emg_chin_tone', 'emg_tone_variance', 'eog_movement_density', 'eog_amplitude', 'heart_rate_mean', 'heart_rate_variability', 'respiration_rate', 'respiration_variability', 'spo2_mean', 'body_movement_index', 'eog_burst_index', 'eog_burst_index_missing', 'delta_minus_theta', 'theta_minus_alpha', 'slowosc_minus_delta', 'beta_minus_delta', 'dt_minus_ab', 'eeg_total']
columns with NaN (expected just EOG): ['eog_burst_index']
